Data Preprocessing and Aggregation

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import timedelta, datetime

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Change working directory to project folder
try:
    os.chdir("/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence")
    print("Directory changed")
except OSError:
    print("Error: Can't change the Current Working Directory")

Mounted at /content/drive
Directory changed


In [ ]:
df_original = pd.read_csv('data/food_waste_augmented.csv')
print(f"Original data shape: {df_original.shape}")

Original data shape: (12587, 27)


In [ ]:
df_original.head()

,Date,Meal,Canteen_Section,Food_Category,Waste_Weight_kg,Unit_Price_per_kg,Cost_Loss,Year,Month,Day,...,Is_High_Waste,Is_High_Cost,Month_Name,Weekday_Type,Waste_per_Price,Log_Waste,DayOfWeek,Hour,Minute,Foot_Traffic
0,2015-01-01 12:00:00,Lunch,B,Soup,2.81,1.5,4.21,2015,1,1,...,0,0,January,Weekday,1.872012,1.032479,NaN,12,0,144.0
1,2015-01-01 13:05:00,Lunch,C,Vegetables,3.29,3.0,9.87,2015,1,1,...,0,0,January,Weekday,1.096722,1.190938,NaN,13,5,96.0
2,2015-01-01 17:14:00,Dinner,A,Vegetables,2.85,3.0,8.55,2015,1,1,...,0,0,January,Weekday,0.949821,1.047130,NaN,17,14,100.0
3,2015-01-01 17:23:00,Dinner,A,Vegetables,0.97,3.0,2.91,2015,1,1,...,0,0,January,Weekday,0.323198,-0.030877,NaN,17,23,100.0
4,2015-01-01 17:46:00,Dinner,B,Vegetables,5.82,3.0,17.45,2015,1,1,...,1,0,January,Weekday,1.938564,1.760560,NaN,17,46,120.0


## 1. Parse Timestamps & Create 30-Minute Bins

In [ ]:
df = df_original.copy()

# Parse Date column to datetime
df['datetime'] = pd.to_datetime(df['Date'])

# Floor to nearest 30-minute bin
df['time_bin'] = df['datetime'].dt.floor('30min')

print(f"Date range: {df['time_bin'].min()} → {df['time_bin'].max()}")
print(f"Sample time bins:\n{df[['Date', 'datetime', 'time_bin']].head(10)}")

Date range: 2015-01-01 12:00:00 → 2025-08-10 19:30:00
Sample time bins:
                  Date            datetime            time_bin
0  2015-01-01 12:00:00 2015-01-01 12:00:00 2015-01-01 12:00:00
1  2015-01-01 13:05:00 2015-01-01 13:05:00 2015-01-01 13:00:00
2  2015-01-01 17:14:00 2015-01-01 17:14:00 2015-01-01 17:00:00
3  2015-01-01 17:23:00 2015-01-01 17:23:00 2015-01-01 17:00:00
4  2015-01-01 17:46:00 2015-01-01 17:46:00 2015-01-01 17:30:00
5  2015-01-02 06:03:00 2015-01-02 06:03:00 2015-01-02 06:00:00
6  2015-01-02 12:03:00 2015-01-02 12:03:00 2015-01-02 12:00:00
7  2015-01-03 11:37:00 2015-01-03 11:37:00 2015-01-03 11:30:00
8  2015-01-04 06:51:00 2015-01-04 06:51:00 2015-01-04 06:30:00
9  2015-01-04 12:03:00 2015-01-04 12:03:00 2015-01-04 12:00:00


## 2. Aggregate per (Canteen Section, Time Bin)

In [ ]:
agg_dict = {
    'Waste_Weight_kg': 'sum',   # total waste per bin
    'Cost_Loss': 'sum',         # total cost loss per bin
    'Foot_Traffic': 'mean'      # average foot traffic per bin
}

df_agg = df.groupby(['Canteen_Section', 'time_bin']).agg(agg_dict).reset_index()

print(f"Aggregated shape: {df_agg.shape}")
print(f"Sections: {df_agg['Canteen_Section'].unique()}")
df_agg.head()

Aggregated shape: (11851, 5)
Sections: ['A' 'B' 'C' 'D']


,Canteen_Section,time_bin,Waste_Weight_kg,Cost_Loss,Foot_Traffic
0,A,2015-01-01 17:00:00,3.82,11.46,100.0
1,A,2015-01-03 11:30:00,6.85,10.27,84.0
2,A,2015-01-04 12:00:00,2.08,4.15,84.0
3,A,2015-01-05 11:30:00,3.61,7.22,120.0
4,A,2015-01-07 06:00:00,0.69,5.48,80.0


## 3. Remove Zero‑Waste Bins (No Artificial Filling)

In [ ]:
# Remove rows where no waste was recorded (Waste_Weight_kg == 0).
# This eliminates the large number of zero‑value entries that were previously
# added by the full reindexing step, keeping only bins that actually contain waste.
rows_before = len(df_agg)
df_agg = df_agg[df_agg['Waste_Weight_kg'] > 0].reset_index(drop=True)
rows_after = len(df_agg)
print(f"Rows before filter: {rows_before:,}")
print(f"Rows after filter : {rows_after:,} ({rows_before - rows_after:,} removed)")

# Note: The dataset now contains only time bins where waste was generated.
# There will be gaps in the time series, which the later feature engineering
# step (lags, rolling windows) will handle by shifting over the irregular sequence.
# This is intentional to reduce dataset size and avoid storing meaningless zeros.

df_agg.head()

Rows before filter: 11,851
Rows after filter : 11,851 (0 removed)


,Canteen_Section,time_bin,Waste_Weight_kg,Cost_Loss,Foot_Traffic
0,A,2015-01-01 17:00:00,3.82,11.46,100.0
1,A,2015-01-03 11:30:00,6.85,10.27,84.0
2,A,2015-01-04 12:00:00,2.08,4.15,84.0
3,A,2015-01-05 11:30:00,3.61,7.22,120.0
4,A,2015-01-07 06:00:00,0.69,5.48,80.0


## 4. Add Time-Based & Cyclical Features

In [ ]:
tb = df_agg['time_bin']

# Raw time features
df_agg['hour']        = tb.dt.hour
df_agg['minute']      = tb.dt.minute
df_agg['weekday']     = tb.dt.dayofweek          # 0=Monday, 6=Sunday
df_agg['month']       = tb.dt.month
df_agg['day_of_year'] = tb.dt.dayofyear
df_agg['quarter']     = tb.dt.quarter
df_agg['is_weekend']  = (tb.dt.dayofweek >= 5).astype(int)

# Cyclical encoding
df_agg['sin_hour']    = np.sin(2 * np.pi * df_agg['hour']    / 24)
df_agg['cos_hour']    = np.cos(2 * np.pi * df_agg['hour']    / 24)
df_agg['sin_minute']  = np.sin(2 * np.pi * df_agg['minute']  / 60)
df_agg['cos_minute']  = np.cos(2 * np.pi * df_agg['minute']  / 60)
df_agg['sin_weekday'] = np.sin(2 * np.pi * df_agg['weekday'] / 7)
df_agg['cos_weekday'] = np.cos(2 * np.pi * df_agg['weekday'] / 7)
df_agg['sin_month']   = np.sin(2 * np.pi * df_agg['month']   / 12)
df_agg['cos_month']   = np.cos(2 * np.pi * df_agg['month']   / 12)

print(f"Final shape: {df_agg.shape}")
print(f"Columns: {list(df_agg.columns)}")
df_agg.head()

Final shape: (11851, 20)
Columns: ['Canteen_Section', 'time_bin', 'Waste_Weight_kg', 'Cost_Loss', 'Foot_Traffic', 'hour', 'minute', 'weekday', 'month', 'day_of_year', 'quarter', 'is_weekend', 'sin_hour', 'cos_hour', 'sin_minute', 'cos_minute', 'sin_weekday', 'cos_weekday', 'sin_month', 'cos_month']


,Canteen_Section,time_bin,Waste_Weight_kg,Cost_Loss,Foot_Traffic,hour,minute,weekday,month,day_of_year,quarter,is_weekend,sin_hour,cos_hour,sin_minute,cos_minute,sin_weekday,cos_weekday,sin_month,cos_month
0,A,2015-01-01 17:00:00,3.82,11.46,100.0,17,0,3,1,1,1,0,-9.659258e-01,-2.588190e-01,0.000000e+00,1.0,0.433884,-0.900969,0.5,0.866025
1,A,2015-01-03 11:30:00,6.85,10.27,84.0,11,30,5,1,3,1,1,2.588190e-01,-9.659258e-01,5.665539e-16,-1.0,-0.974928,-0.222521,0.5,0.866025
2,A,2015-01-04 12:00:00,2.08,4.15,84.0,12,0,6,1,4,1,1,1.224647e-16,-1.000000e+00,0.000000e+00,1.0,-0.781831,0.623490,0.5,0.866025
3,A,2015-01-05 11:30:00,3.61,7.22,120.0,11,30,0,1,5,1,0,2.588190e-01,-9.659258e-01,5.665539e-16,-1.0,0.000000,1.000000,0.5,0.866025
4,A,2015-01-07 06:00:00,0.69,5.48,80.0,6,0,2,1,7,1,0,1.000000e+00,6.123234e-17,0.000000e+00,1.0,0.974928,-0.222521,0.5,0.866025


## 5. Save Preprocessed Data

In [ ]:
output_path = 'data/food_waste_preprocessed.csv'
df_agg.to_csv(output_path, index=False)
print(f"Saved to {output_path}  |  shape: {df_agg.shape}")

Saved to data/food_waste_preprocessed.csv  |  shape: (11851, 20)
